# 📊 Process data: A0–A3

Chuẩn hóa sentinel, tạo feature, thống kê mô tả và kiểm tra chất lượng dữ liệu. `task_type`, `ai_tool`, `ai_verb` là heuristic.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
DATA_PATH = Path('../data/processed/cleaned_commits_data_final.csv')
if not DATA_PATH.exists(): DATA_PATH = Path('../data/interim/latest_standardized_data.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Đã tải {DATA_PATH}: {len(df):,} dòng × {df.shape[1]} cột')

Đã tải ..\data\processed\cleaned_commits_data_final.csv: 33,917 dòng × 10 cột


## A0 — Sửa sentinel và chuẩn hóa kiểu ngày
`Not Modified`/`No URL` được chuyển thành missing chuẩn; `is_modified` giữ lại ý nghĩa của dữ liệu thiếu có chủ đích.

In [2]:
for col in ['change_commit_date', 'change_commit_url']:
    if col in df.columns: df[col] = df[col].replace({'Not Modified': pd.NA, 'No URL': pd.NA})
for col in ['commit_date', 'change_commit_date']:
    if col in df.columns: df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
df['is_modified'] = df['change_commit_date'].notna()
print(df[['commit_date', 'change_commit_date', 'change_commit_url', 'is_modified']].isna().sum())

commit_date               0
change_commit_date    21327
change_commit_url     21327
is_modified               0
dtype: int64


## A1 — Feature engineering

In [3]:
df_clean = df.dropna(subset=['commit_date']).copy()
df_clean['year'] = df_clean['commit_date'].dt.year.astype('Int64')
df_clean['month'] = df_clean['commit_date'].dt.month.astype('Int64')
df_clean['year_month'] = df_clean['commit_date'].dt.strftime('%Y-%m')
df_clean['quarter'] = 'Q' + df_clean['commit_date'].dt.quarter.astype(str)
df_clean['hour'] = df_clean['commit_date'].dt.hour.astype('Int64')
df_clean['day_of_week'] = df_clean['commit_date'].dt.day_name()
df_clean['is_weekend'] = df_clean['commit_date'].dt.dayofweek >= 5
df_clean['days_to_modification'] = ((df_clean['change_commit_date'] - df_clean['commit_date']).dt.total_seconds() / 86400).round(3)
negative_days = int((df_clean['days_to_modification'] < 0).sum())
df_clean.loc[df_clean['days_to_modification'] < 0, 'days_to_modification'] = np.nan
code = df_clean['code_block'].fillna('').astype(str)
# code_block lưu newline dưới dạng ký tự escape \\n; vẫn hỗ trợ newline thật nếu có
df_clean['code_lines'] = code.str.count(r'\\n').add(code.str.count('\n')).add(code.ne('').astype(int))
df_clean['code_chars'] = code.str.len()
text = (df_clean.get('id', pd.Series('', index=df_clean.index)).fillna('').astype(str) + ' ' + code).str.lower()
def first_label(series, rules, default='unknown'):
    result = pd.Series(default, index=series.index, dtype='object')
    for label, pattern in rules:
        result.loc[result.eq(default) & series.str.contains(pattern, regex=True, na=False)] = label
    return result
df_clean['ai_tool'] = first_label(text, [('chatgpt_gpt', r'chatgpt|gpt[-_ ]?3|gpt[-_ ]?4|written_by_gpt'), ('github_copilot', r'copilot'), ('claude', r'\bclaude\b'), ('gemini_bard', r'gemini|bard'), ('deepseek', r'deepseek'), ('codeium', r'codeium'), ('cursor', r'\bcursor\b'), ('other_ai', r'\bai\b|artificial intelligence')])
df_clean['ai_verb'] = first_label(text, [('written', r'written by|write(?:n)? by|generated by|created by'), ('fixed', r'fixed by|bug fix|fix(?:ed)? with'), ('refactored', r'refactor(?:ed)?|refactoring'), ('tested', r'test(?:s|ed|ing)? by|unit test'), ('assisted', r'assist(?:ed)? by|help(?:ed)? by|suggested by|using ai')])
df_clean['repo_commit_count'] = df_clean.groupby('repo')['commit_url'].transform('nunique')
df_clean['task_type'] = first_label(text, [('testing', r'\btest(?:s|ed|ing)?\b|pytest|jest|spec'), ('documentation', r'docstring|documentation|readme|\bdocs?\b|comment'), ('bug_fix', r'bug|error|exception|fix(?:ed|es)?'), ('refactoring', r'refactor|rename|cleanup|clean[ -]?up'), ('data_query', r'sql|query|database|\bapi\b'), ('feature_implementation', r'implement|function|class|feature|create|add ')])
print(f'Đã loại {len(df) - len(df_clean):,} dòng thiếu commit_date')
print(f'Có {negative_days:,} dòng days_to_modification âm; đã chuyển thành NaN cho phân tích days')
display(df_clean.head(3))

Đã loại 0 dòng thiếu commit_date
Có 2,115 dòng days_to_modification âm; đã chuyển thành NaN cho phân tích days


,id,repo,language,commit_url,file,code_block,commit_date,change_commit_url,change_commit_date,is_modified,year,month,year_month,quarter,hour,day_of_week,is_weekend,days_to_modification,code_lines,code_chars,ai_tool,ai_verb,repo_commit_count,task_type
0,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,2022-12-03 08:41:16+00:00,NaN,NaT,False,2022,12,2022-12,Q4,8,Saturday,True,NaN,18,543,chatgpt_gpt,written,1,feature_implementation
1,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,2022-12-03 08:41:16+00:00,NaN,NaT,False,2022,12,2022-12,Q4,8,Saturday,True,NaN,18,543,chatgpt_gpt,written,1,feature_implementation
2,python__written_by_ai__437874902__14f89ee997ec...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle2.py,# Written by AI\n\ndef spreadsheet_checksum(li...,2022-12-04 05:58:44+00:00,NaN,NaT,False,2022,12,2022-12,Q4,5,Sunday,True,NaN,10,221,other_ai,written,1,unknown


## A2 — Descriptive statistics

In [4]:
numeric_cols = ['days_to_modification', 'code_lines', 'code_chars', 'repo_commit_count']
description = df_clean[numeric_cols].describe().T
description['skew'] = df_clean[numeric_cols].skew()
description['kurtosis'] = df_clean[numeric_cols].kurtosis()
display(description)
outlier_rows = []
for col in numeric_cols:
    s = df_clean[col].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3-q1; lo, hi = q1-1.5*iqr, q3+1.5*iqr
    outlier_rows.append({'variable': col, 'q1': q1, 'q3': q3, 'lower_bound': lo, 'upper_bound': hi, 'outlier_count': int(((s < lo) | (s > hi)).sum())})
display(pd.DataFrame(outlier_rows))
for col in ['language', 'ai_tool', 'ai_verb', 'task_type', 'is_modified']:
    print(f'\nTần suất: {col}')
    display(df_clean[col].value_counts(dropna=False).rename_axis(col).reset_index(name='count').head(20))
print('Crosstab language × is_modified:')
display(pd.crosstab(df_clean['language'], df_clean['is_modified'], margins=True))

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
days_to_modification,10475.0,23.708321,65.658111,0.0,0.027,1.284,13.8915,1011.911,5.654644,43.678649
code_lines,33917.0,95.434001,187.233638,2.0,12.000,32.000,87.0000,2833.000,4.782710,31.895673
code_chars,33917.0,3635.713212,7448.496940,2.0,397.000,1145.000,3247.0000,101285.000,4.940564,33.405481
repo_commit_count,33917.0,13.846242,41.197420,1.0,1.000,1.000,2.0000,159.000,3.219821,8.436969


,variable,q1,q3,lower_bound,upper_bound,outlier_count
0,days_to_modification,0.027,13.8915,-20.76975,34.68825,1696
1,code_lines,12.000,87.0000,-100.50000,199.50000,4157
2,code_chars,397.000,3247.0000,-3878.00000,7522.00000,4190
3,repo_commit_count,1.000,2.0000,-0.50000,3.50000,6332



Tần suất: language


,language,count
0,python,23844
1,js,10073



Tần suất: ai_tool


...

,task_type,count
0,testing,9762
1,feature_implementation,7313
2,unknown,6608
3,bug_fix,5981
4,data_query,2088
5,documentation,1918
6,refactoring,247



Tần suất: is_modified


,is_modified,count
0,False,21327
1,True,12590


Crosstab language × is_modified:


is_modified,False,True,All
language,,,
js,6578,3495,10073
python,14749,9095,23844
All,21327,12590,33917


## A3 — Data quality checks
Ngày sửa âm được giữ ở `is_modified=True` nhưng loại khỏi `days_to_modification`; outlier chỉ được báo cáo, không tự động xóa.

In [5]:
quality = {'rows': len(df_clean), 'columns': df_clean.shape[1], 'duplicate_rows': int(df_clean.duplicated().sum()), 'duplicate_ids': int(df_clean['id'].duplicated().sum()), 'negative_days_original': negative_days, 'missing_commit_date': int(df_clean['commit_date'].isna().sum()), 'commit_date_min': df_clean['commit_date'].min(), 'commit_date_max': df_clean['commit_date'].max(), 'invalid_days_after_cleaning': int((df_clean['days_to_modification'] < 0).sum())}
display(pd.Series(quality, name='value'))
assert df_clean['commit_date'].notna().all()
assert df_clean['is_modified'].dtype == bool
assert (df_clean['days_to_modification'].dropna() >= 0).all()
output_path = Path('../data/processed/commits_features.csv')
df_clean.to_csv(output_path, index=False)
print(f'✅ Đã xuất {output_path} ({output_path.stat().st_size / 1024 / 1024:.1f} MB)')
print(f'Shape: {df_clean.shape[0]:,} dòng × {df_clean.shape[1]} cột')

rows                                               33917
columns                                               24
duplicate_rows                                         0
duplicate_ids                                          0
negative_days_original                              2115
missing_commit_date                                    0
commit_date_min                2022-12-03 08:41:16+00:00
commit_date_max                2026-04-10 16:29:46+00:00
invalid_days_after_cleaning                            0
Name: value, dtype: object

✅ Đã xuất ..\data\processed\commits_features.csv (142.4 MB)
Shape: 33,917 dòng × 24 cột
